In [1]:
import os, time
import pandas as pd, requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))     
API_KEY = os.getenv("OPENAQ_API_KEY")
assert API_KEY, "OPENAQ_API_KEY not found. Check that .env is in the project folder."
BASE = "https://api.openaq.org"

def get(path, **params):
    for attempt in range(5):
        r = requests.get(BASE + path, params=params, headers={"X-API-Key": API_KEY}, timeout=60)
        time.sleep(1.1)                         
        if r.status_code == 429:              
            wait = int(r.headers.get("x-ratelimit-reset", 60))
            print(f"Rate limit reached, waiting {wait} s ...")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"Gave up on {path} after 5 attempts")

countries = pd.DataFrame(get("/v3/countries", limit=300)["results"])
print("Countries with data on OpenAQ:", len(countries))
west_africa = ["NG", "GH", "BJ", "TG", "CI", "SN", "NE", "CM", "BF", "ML"]
wa = countries[countries.code.isin(west_africa)].copy()
wa["pollutants"] = wa.parameters.apply(lambda ps: ", ".join(sorted({p["name"] for p in ps})))
wa["first"] = wa.datetimeFirst.str[:10]; wa["last"] = wa.datetimeLast.str[:10]
print(wa[["id", "code", "name", "first", "last", "pollutants"]].to_string(index=False))

Countries with data on OpenAQ: 159
 id code          name      first       last                                                         pollutants
 96   CI Côte d'Ivoire 2020-03-15 2026-10-10 co, no2, o3, pm1, pm10, pm25, relativehumidity, temperature, um003
 98   ML          Mali 2020-03-16 2025-03-04                                                               pm25
 99   SN       Senegal 2023-09-05 2026-10-10              pm1, pm10, pm25, relativehumidity, temperature, um003
100   NG       Nigeria 2019-04-26 2026-10-10    pm1, pm10, pm25, pressure, relativehumidity, temperature, um003
147   CM      Cameroon 2024-09-23 2026-10-10              pm1, pm10, pm25, relativehumidity, temperature, um003
149   NE         Niger 2026-06-01 2026-06-01                    pm1, pm25, relativehumidity, temperature, um003
150   BF  Burkina Faso 2026-04-30 2026-10-10                     no2, pm10, pm25, relativehumidity, temperature
152   GH         Ghana 2020-05-01 2026-10-10         no2, pm1, pm10, 

In [2]:
from datetime import datetime, timedelta, timezone

NIGERIA_BBOX = "2.67,4.27,14.68,13.89"                  

rows, page = [], 1
while True:
    data = get("/v3/locations", bbox=NIGERIA_BBOX, limit=1000, page=page)
    for loc in data["results"]:
        rows.append({
            "location_id": loc["id"], "name": loc["name"], "city": loc.get("locality"),
            "country": loc["country"]["code"], "provider": loc["provider"]["name"],
            "owner": (loc.get("owner") or {}).get("name"), "is_monitor": loc.get("isMonitor"), "mobile": loc.get("isMobile"),
            "pollutants": ", ".join(sorted({s["parameter"]["name"] for s in loc["sensors"]})), "sensors": len(loc["sensors"]),
            "first": (loc.get("datetimeFirst") or {}).get("utc"), "last": (loc.get("datetimeLast") or {}).get("utc"),
            "lat": loc["coordinates"]["latitude"], "lon": loc["coordinates"]["longitude"]})
    if len(data["results"]) < 1000:
        break
    page += 1

locs = pd.DataFrame(rows)
locs["first"] = pd.to_datetime(locs["first"], utc=True)
locs["last"] = pd.to_datetime(locs["last"], utc=True)
ng = locs[locs.country == "NG"].copy()
recent = ng[ng["last"] >= datetime.now(timezone.utc) - timedelta(days=30)]

print("Locations found in the Nigeria box:", len(locs), "| in Nigeria:", len(ng), "| reported in the last 30 days:", len(recent))
print("\nBy provider (all Nigerian locations / active in last 30 days):")
print(pd.concat([ng.provider.value_counts(), recent.provider.value_counts()], axis=1, keys=["all", "active"]).fillna(0).astype(int).to_string())
print("\nBy city (active only):"); print(recent.city.value_counts().head(15).to_string())
print("\nPollutants measured (active locations):"); print(recent.pollutants.str.split(", ").explode().value_counts().to_string())
print("\nThe 25 longest-running active locations:")
recent = recent.assign(years=((recent["last"] - recent["first"]).dt.days / 365.25).round(1))
print(recent.sort_values("first")[["location_id", "name", "city", "provider", "is_monitor", "pollutants", "years"]].head(25).to_string(index=False))

os.makedirs("../data", exist_ok=True)
locs.to_csv("../data/locations_nigeria_box.csv", index=False)

Locations found in the Nigeria box: 102 | in Nigeria: 101 | reported in the last 30 days: 65

By provider (all Nigerian locations / active in last 30 days):
                 all  active
provider                    
AirGradient       58      36
AirQo             17      16
Miri Africa       10       4
HabitatMap         6       5
IQAir              3       3
AirNow             2       0
Clarity            2       0
Spartan            1       0
SPARTAN Network    1       0
Aurassure          1       1

By city (active only):
Series([], )

Pollutants measured (active locations):
pollutants
pm25                65
relativehumidity    41
temperature         41
pm1                 36
um003               36
pm10                21
pressure             4

The 25 longest-running active locations:
 location_id                                                   name city    provider  is_monitor                                          pollutants  years
     4903422                                 Do

In [3]:
import numpy as np

CITIES = {"Lagos": (6.52, 3.38), "Abuja": (9.07, 7.49), "Ibadan": (7.38, 3.94), "Abeokuta": (7.16, 3.35),
          "Kano": (12.00, 8.52), "Kaduna": (10.52, 7.44), "Port Harcourt": (4.82, 7.03), "Benin City": (6.34, 5.63),
          "Enugu": (6.45, 7.51), "Jos": (9.90, 8.86), "Ilorin": (8.50, 4.55), "Maiduguri": (11.85, 13.16)}

def km_between(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def nearest_city(lat, lon, max_km=60):
    name, km = min(((c, km_between(lat, lon, *xy)) for c, xy in CITIES.items()), key=lambda t: t[1])
    return name if km <= max_km else f"Other (nearest {name}, {km:.0f} km)"

ng["city"] = [nearest_city(a, b) for a, b in zip(ng.lat, ng.lon)]        
ng["active"] = ng["last"] >= datetime.now(timezone.utc) - timedelta(days=30)

sensor_rows, page = [], 1                                                 
while True:
    data = get("/v3/locations", bbox=NIGERIA_BBOX, limit=1000, page=page)
    for loc in data["results"]:
        for s in loc["sensors"]:
            sensor_rows.append({"location_id": loc["id"], "sensor_id": s["id"], "parameter": s["parameter"]["name"], "units": s["parameter"]["units"]})
    if len(data["results"]) < 1000:
        break
    page += 1
sensors = pd.DataFrame(sensor_rows).merge(ng[["location_id", "name", "city", "provider", "active"]], on="location_id")

print("Active locations by city and provider:")
print(pd.crosstab(ng[ng.active].city, ng[ng.active].provider, margins=True, margins_name="total").to_string())
print("\nSensors at active locations, by what they measure and in which units:")
print(sensors[sensors.active].groupby(["parameter", "units"]).size().to_string())

ng.to_csv("../data/locations_nigeria.csv", index=False)
sensors.to_csv("../data/sensors_nigeria.csv", index=False)

Active locations by city and provider:
provider       AirGradient  AirQo  Aurassure  HabitatMap  IQAir  Miri Africa  total
city                                                                               
Abuja                    7      1          0           0      0            0      8
Enugu                    1      0          0           0      0            0      1
Ibadan                   7      1          0           0      0            0      8
Lagos                   19     10          1           5      3            4     42
Port Harcourt            2      4          0           0      0            0      6
total                   36     16          1           5      3            4     65

Sensors at active locations, by what they measure and in which units:
parameter         units        
pm1               µg/m³            36
pm10              µg/m³            21
pm25              µg/m³            65
pressure          hpa               4
relativehumidity  %               

In [4]:
DAYS = 60
end = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start = end - timedelta(days=DAYS)

pm = sensors[(sensors.parameter == "pm25") & sensors.active]
sample = pm.sample(frac=1, random_state=42).groupby("provider").head(3)         
print("Sample:", len(sample), "PM2.5 sensors from", sample.provider.nunique(), "providers")

rows = []
for s in sample.itertuples():
    page = 1
    while True:
        data = get(f"/v3/sensors/{s.sensor_id}/hours", datetime_from=start.isoformat(), datetime_to=end.isoformat(), limit=1000, page=page)
        for h in data["results"]:
            cov = h.get("coverage") or {}
            rows.append({"sensor_id": s.sensor_id, "provider": s.provider, "city": s.city,
                         "hour_utc": h["period"]["datetimeFrom"]["utc"], "value": h["value"],
                         "observed": cov.get("observedCount"), "expected": cov.get("expectedCount"), "coverage_pct": cov.get("percentCoverage")})
        if len(data["results"]) < 1000:
            break
        page += 1

hours = pd.DataFrame(rows)
hours["hour_utc"] = pd.to_datetime(hours.hour_utc, utc=True)
os.makedirs("../data/raw", exist_ok=True)
hours.to_csv("../data/raw/sample_pm25_hours.csv", index=False)

def longest_repeat(v):
    runs = (v != v.shift()).cumsum()
    return int(v.groupby(runs).size().max())

per_sensor = hours.sort_values("hour_utc").groupby(["provider", "city", "sensor_id"]).agg(
    hours=("value", "size"), median=("value", "median"), max=("value", "max"),
    zero_or_negative=("value", lambda v: int((v <= 0).sum())), above_500=("value", lambda v: int((v > 500).sum())),
    longest_repeat=("value", longest_repeat), median_coverage=("coverage_pct", "median"))
per_sensor["share_of_hours_present"] = (per_sensor.hours / (DAYS * 24)).round(2)
print(f"\nHourly PM2.5, {start:%Y-%m-%d} to {end:%Y-%m-%d} ({DAYS * 24} hours possible):")
print(per_sensor.round(1).to_string())
print("\nDuplicate hours for the same sensor:", int(hours.duplicated(["sensor_id", "hour_utc"]).sum()))

Sample: 16 PM2.5 sensors from 6 providers

Hourly PM2.5, 2026-08-11 to 2026-10-10 (1440 hours possible):
                                     hours  median    max  zero_or_negative  above_500  longest_repeat  median_coverage  share_of_hours_present
provider    city          sensor_id                                                                                                            
AirGradient Abuja         17487012     891    28.8   84.0                 0          0               2            100.0                     0.6
            Lagos         13578919    1402    27.7  527.0                 0          1               2            100.0                     1.0
                          13844885     820    21.2   69.2                 0          0               2            100.0                     0.6
AirQo       Lagos         16166140     427    22.1  181.0                 0          0               2            100.0                     0.3
                          16350

In [5]:
day = hours.hour_utc.dt.floor("D")
hours_per_day = hours.groupby(["provider", "city", "sensor_id", day]).size()

gaps = []
for (prov, city, sid), g in hours.groupby(["provider", "city", "sensor_id"]):
    t = pd.concat([pd.Series([pd.Timestamp(start) - pd.Timedelta(hours=1)]), g.hour_utc.sort_values(), pd.Series([pd.Timestamp(end)])], ignore_index=True)
    missing = (t.diff().dt.total_seconds() / 3600 - 1).dropna()
    missing = missing[missing > 0]
    usable = int((hours_per_day.loc[(prov, city, sid)] >= 18).sum())
    gaps.append({"provider": prov, "city": city, "sensor_id": sid, "gaps": len(missing),
                 "longest_gap_hours": int(missing.max()) if len(missing) else 0,
                 "single_missing_hours": int((missing == 1).sum()), "usable_days": usable})
gaps = pd.DataFrame(gaps).set_index(["provider", "city", "sensor_id"])
gaps["usable_share"] = (gaps.usable_days / DAYS).round(2)
print(f"Missing hours, and days with at least 18 of 24 hours (out of {DAYS} days):")
print(gaps.to_string())

Missing hours, and days with at least 18 of 24 hours (out of 60 days):
                                     gaps  longest_gap_hours  single_missing_hours  usable_days  usable_share
provider    city          sensor_id                                                                          
AirGradient Abuja         17487012      2                548                     1           36          0.60
            Lagos         13578919     20                  3                     7           59          0.98
                          13844885     15                304                     0           32          0.53
AirQo       Lagos         16166140    248                 35                   106            0          0.00
                          16350109    196                 59                    80            0          0.00
            Port Harcourt 16166146    243                 39                   107            0          0.00
Aurassure   Lagos         16577537    107        

In [6]:
lagos = hours[hours.city == "Lagos"]
compare = []
for sid, g in lagos.groupby("sensor_id"):
    others = lagos[lagos.sensor_id != sid].groupby("hour_utc").value.agg(["median", "size"])
    m = g.set_index("hour_utc").join(others, how="inner")
    m = m[m["size"] >= 3]
    ratio = m.value / m["median"]
    compare.append({"provider": g.provider.iloc[0], "sensor_id": sid, "hours_compared": len(m),
                    "median_ratio": round(ratio.median(), 2),
                    "share_over_3x": round((ratio > 3).mean(), 3), "share_under_third": round((ratio < 1/3).mean(), 3)})
print("Each Lagos sensor against the median of the other Lagos sensors in the same hour:")
print(pd.DataFrame(compare).set_index(["provider", "sensor_id"]).to_string())

aur = hours[hours.provider == "Aurassure"].value
print("\nAurassure values at chosen percentiles:")
print(aur.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1).to_string())
print("Hours at exactly its maximum value:", int((aur == aur.max()).sum()), "| distinct values above 500:", sorted(aur[aur > 500].unique()))

spike = hours.loc[hours.value[hours.provider != "Aurassure"].idxmax()]
around = hours[(hours.sensor_id == spike.sensor_id) & (hours.hour_utc - spike.hour_utc).abs().le(pd.Timedelta(hours=3))]
print(f"\nThe highest reading from any other sensor ({spike.provider}, {spike.sensor_id}), with 3 hours either side:")
print(around[["hour_utc", "value"]].to_string(index=False))

Each Lagos sensor against the median of the other Lagos sensors in the same hour:
                       hours_compared  median_ratio  share_over_3x  share_under_third
provider    sensor_id                                                                
AirGradient 13578919             1390          1.70          0.070              0.001
            13844885              820          1.30          0.023              0.005
IQAir       13923456             1345          0.84          0.006              0.038
            13923457             1281          0.99          0.022              0.008
            13923470             1383          1.36          0.031              0.001
Miri Africa 15787512              173          0.61          0.000              0.029
            15787517              151          1.74          0.139              0.000
            15787547               93          0.21          0.000              0.860
AirQo       16166140              427          1.32       

In [7]:
hours["local_hour"] = (hours.hour_utc + pd.Timedelta(hours=1)).dt.hour         
hours["block"] = pd.cut(hours.local_hour, [0, 6, 12, 18, 24], right=False, labels=["00-06", "06-12", "12-18", "18-24"])

present = hours.groupby(["provider", "block"], observed=True).size().unstack()
possible = hours.groupby("provider").sensor_id.nunique() * DAYS * 6
print("Share of possible hours present, by time of day (Nigerian time):")
print(present.div(possible, axis=0).round(2).to_string())

complete_ids = per_sensor[per_sensor.share_of_hours_present >= 0.9].index.get_level_values("sensor_id")
print(f"\nMedian PM2.5 by time of day, from the {len(complete_ids)} near-complete sensors:")
print(hours[hours.sensor_id.isin(complete_ids)].groupby("block", observed=True).value.median().round(1).to_string())

Share of possible hours present, by time of day (Nigerian time):
block        00-06  06-12  12-18  18-24
provider                               
AirGradient   0.73   0.69   0.73   0.74
AirQo         0.06   0.25   0.41   0.36
Aurassure     0.81   0.87   0.81   0.76
HabitatMap    0.02   0.00   0.62   0.84
IQAir         0.95   0.92   0.91   0.96
Miri Africa   0.00   0.02   0.25   0.11

Median PM2.5 by time of day, from the 3 near-complete sensors:
block
00-06    19.8
06-12    27.2
12-18    19.0
18-24    22.0


In [8]:
near = hours[(hours.city == "Lagos") & (hours.provider != "Aurassure") & (hours.sensor_id != spike.sensor_id)
             & (hours.hour_utc - spike.hour_utc).abs().le(pd.Timedelta(hours=3))]
print("Other Lagos sensors during the 527 event:")
print(near.groupby("hour_utc").value.agg(sensors="size", median="median", highest="max").round(1).to_string())

def isolated_spikes(g, factor=5, floor=100):
    v = g.set_index("hour_utc").value.sort_index().asfreq("h")
    around = pd.concat([v.shift(2), v.shift(1), v.shift(-1), v.shift(-2)], axis=1).median(axis=1)
    return v[(v > floor) & (v > factor * around)]

print("\nCandidate rule: flag an hour above 100 that is more than 5 times the median of the 2 hours either side")
for (prov, sid), g in hours.groupby(["provider", "sensor_id"]):
    s = isolated_spikes(g)
    print(f"{prov:12} {sid:>9}: {len(s):3} flagged", "| e.g. " + ", ".join(f"{t:%m-%d %H:00} = {x:.0f}" for t, x in s.head(3).items()) if len(s) else "")

Other Lagos sensors during the 527 event:
                           sensors  median  highest
hour_utc                                           
2026-08-21 03:00:00+00:00        4    19.9     28.0
2026-08-21 04:00:00+00:00        4    24.1     32.0
2026-08-21 05:00:00+00:00        4    25.5     32.5
2026-08-21 06:00:00+00:00        4    26.6     33.0
2026-08-21 07:00:00+00:00        4    27.5     28.1
2026-08-21 08:00:00+00:00        4    24.5     27.0
2026-08-21 09:00:00+00:00        4    24.4     28.0

Candidate rule: flag an hour above 100 that is more than 5 times the median of the 2 hours either side
AirGradient   13578919:   0 flagged 
AirGradient   13844885:   0 flagged 
AirGradient   17487012:   0 flagged 
AirQo         16166140:   0 flagged 
AirQo         16166146:   0 flagged 
AirQo         16350109:   3 flagged | e.g. 08-18 14:00 = 173, 09-15 18:00 = 174, 10-06 19:00 = 105
Aurassure     16577537:  26 flagged | e.g. 08-13 14:00 = 694, 08-13 15:00 = 682, 08-15 15:00 = 519
Hab